# Harley 1992: photosynthesis of cotton grown in elevated CO₂

Reference: P. C. Harley, R. B. Thomas, J. F. Reynolds, B. R. Strain
"Modelling photosynthesis of cotton grown in elevated CO2"
Plant, Cell & Environment 15 (1992): 271-282.

[doi: 10.1111/j.1365-3040.1992.tb00974.x](https://www.doi.org/10.1111/j.1365-3040.1992.tb00974.x)

The model is evaluated with the plain numpy implementation below, independent
of green-sloth's `model.ts`. Ci is the driving variable; the ambient CO₂ (Ca)
and stomatal conductance follow from Eqns 10 and 11. Figs 4-7 compare
regressions or predictions with measured data and are not reproduced.

O₂ is 20 kPa, the O₂ of the measurements (Fig 1, Table 2): with it the model
reproduces the operating points printed in Fig 8 exactly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq

R = 0.00831  # kJ K^-1 mol^-1, as in the paper
# Table 1: temperature parameters (kJ mol^-1) for plants grown in 35 and 65 Pa CO2
SET35 = dict(c_V=51.07, dHa_V=116.3, dHd_V=202.9, c_J=37.08, dHa_J=79.5, dHd_J=201.0,
             c_T=23.59, dHa_T=53.1, dHd_T=201.8, g0=81.1, g1=9.58, c_Rdark=34.13, dHa_Rdark=84.45)
SET65 = dict(c_V=49.46, dHa_V=112.4, dHd_V=201.7, c_J=32.88, dHa_J=69.1, dHd_J=201.1,
             c_T=23.59, dHa_T=53.1, dHd_T=201.6, g0=72.8, g1=8.92, c_Rdark=29.96, dHa_Rdark=73.45)


def eqn8(c, dHa, Tk):
    return np.exp(c - dHa / (R * Tk))


def eqn9(c, dHa, dHd, Tk, dS=0.65):
    return eqn8(c, dHa, Tk) / (1 + np.exp((dS * Tk - dHd) / (R * Tk)))


def harley(Ci=25.9, O=20.0, T_leaf=29.0, PPFD=1000.0, RH=50.0, Rd=0.82, alpha=0.24,
           Vcmax=None, Jmax=None, TPU=None, **params) -> pd.DataFrame:
    """One scanned condition (an array) indexes the result; capacities may be fixed."""
    p = SET35 | params
    (name, x), = ((k, v) for k, v in dict(Ci=Ci, O=O, T_leaf=T_leaf, PPFD=PPFD).items() if np.ndim(v) == 1)
    Ci, O, T_leaf, PPFD = np.broadcast_arrays(*(np.asarray(v, dtype=float) for v in (Ci, O, T_leaf, PPFD)))
    Tk = T_leaf + 273.15
    Kc, Ko, tau = eqn8(35.79, 80.47, Tk), eqn8(9.59, 14.51, Tk), eqn8(-3.9489, -28.99, Tk)
    Gamma_star = 500 * O / tau  # 0.5 O / tau, O in Pa
    Vcmax = eqn9(p["c_V"], p["dHa_V"], p["dHd_V"], Tk) if Vcmax is None else np.full_like(Tk, Vcmax)
    Jmax = eqn9(p["c_J"], p["dHa_J"], p["dHd_J"], Tk) if Jmax is None else np.full_like(Tk, Jmax)
    TPU = eqn9(p["c_T"], p["dHa_T"], p["dHd_T"], Tk) if TPU is None else np.full_like(Tk, TPU)
    J = alpha * PPFD / np.sqrt(1 + (alpha * PPFD / Jmax) ** 2)  # Eqn 7
    Wc = Vcmax * Ci / (Ci + Kc * (1 + O / Ko))  # Eqn 4
    Wj = J * Ci / (4 * (Ci + 2 * Gamma_star))  # Eqn 5
    co2 = 1 - Gamma_star / Ci
    Wp = 3 * TPU / np.maximum(co2, 1e-12)  # Eqn 6 solved for Wp
    A = co2 * np.minimum.reduce([Wc, Wj, Wp]) - Rd  # Eqn 3
    # Eqns 10 and 11 solved for Ca
    g0, g1 = p["g0"], p["g1"]
    b = g1 * A * RH - (g0 * Ci + 160 * A)
    Ca = (-b + np.sqrt(np.maximum(b**2 + 4 * g0 * g1 * A * RH * Ci, 0))) / (2 * g0)
    gs = g0 + g1 * A * RH / Ca
    return pd.DataFrame(
        {"Vcmax": Vcmax, "Jmax": Jmax, "TPU": TPU, "Gamma_star": Gamma_star, "J": J,
         "rubisco": co2 * Wc - Rd, "rubp": co2 * Wj - Rd, "tpu": 3 * TPU - Rd, "A": A,
         "Ca": Ca, "gs": gs, "R_dark": eqn8(p["c_Rdark"], p["dHa_Rdark"], Tk)},
        index=pd.Index(np.asarray(x, dtype=float), name=name),
    )


def operating_point(Ca, **kw):
    """The Ci at which the supply (Eqns 10, 11) meets the demand (Eqn 3)."""
    Ci = brentq(lambda ci: harley(Ci=np.array([ci]), **kw)["Ca"].iloc[0] - Ca, 5, Ca)
    return harley(Ci=np.array([Ci]), **kw).iloc[0].rename(Ci)

## Figure 1: A/Ci at 2 and 20 kPa O₂

Representative plants grown at 35 and 65 Pa CO₂, measured at 29 °C and PPFD 1500, with the
fitted Vc,max, Rd, Jmax and TPU given in the figure. Dashed: no phosphate limitation.

In [ ]:
FIG1 = {
    "a": {"solid": dict(Vcmax=127.3, Rd=2.12, Jmax=231.4, TPU=12.60), "dashed": dict(Vcmax=127.3, Rd=2.12, Jmax=192.2, TPU=1e6)},
    "b": {"solid": dict(Vcmax=138.7, Rd=1.07, Jmax=228.6, TPU=11.51), "dashed": dict(Vcmax=138.7, Rd=1.07, Jmax=177.8, TPU=1e6)},
}
fig1 = {
    f"{panel}_{line}_{o2}": harley(Ci=np.linspace(1, 70 if panel == "a" else 80, 400), O=o2, T_leaf=29.0, PPFD=1500.0, **fit)
    for panel, lines in FIG1.items() for line, fit in lines.items() for o2 in (20.0, 2.0)
}
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
for ax, panel, title in zip(axs, "ab", ["Cotton, grown at 35 Pa CO$_2$", "Grown at 65 Pa CO$_2$"]):
    for line in ("solid", "dashed"):
        for o2 in (20.0, 2.0):
            res = fig1[f"{panel}_{line}_{o2}"]
            ax.plot(res.index, res["A"], color="black", linestyle="-" if line == "solid" else "--")
    ax.set(title=title, xlabel="Intercellular CO$_2$ (Pa)", xlim=(0, 70 if panel == "a" else 80), ylim=(0, 40))
axs[0].set_ylabel(r"Net Photosynthesis ($\mu$mol m$^{-2}$ s$^{-1}$)")
fig.tight_layout()
fig.savefig("figs/fig1.svg")
plt.show()

## Figure 2: dark respiration

The paper's fits to Eqn 8 for plants grown in 35 and 65 Pa CO₂ (not used by the model, which
holds Rd at 0.82).

In [ ]:
fig2 = {k: harley(T_leaf=np.linspace(12, 36, 241), **s) for k, s in {"35": SET35, "65": SET65}.items()}
fig, ax = plt.subplots(figsize=(4.6, 4.0))
for k, res in fig2.items():
    ax.plot(res.index, res["R_dark"], color="black", label=f"{k} Pa CO$_2$")
ax.set(xlabel="Leaf Temperature (°C)", ylabel=r"Dark Respiration ($\mu$mol m$^{-2}$ s$^{-1}$)", xlim=(12, 36), ylim=(0, 4))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("figs/fig2.svg")
plt.show()

## Figure 3: Vc,max, Jmax and TPU against leaf temperature

Eqn 9 with the parameters of Table 1.

In [ ]:
fig3 = {k: harley(T_leaf=np.linspace(10, 40, 301), **s) for k, s in {"35": SET35, "65": SET65}.items()}
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, ylim in zip(axs, ["Vcmax", "Jmax", "TPU"], [(0, 200), (0, 250), (0, 14)]):
    for k, res in fig3.items():
        ax.plot(res.index, res[col], color="black", linestyle="-" if k == "35" else "--", label=f"{k} Pa CO$_2$")
    ax.set(xlabel="Leaf Temperature (°C)", ylabel=col, xlim=(10, 40), ylim=ylim)
axs[0].legend(frameon=False)
fig.tight_layout()
fig.savefig("figs/fig3.svg")
plt.show()

## Figure 8: net photosynthesis against CO₂

At the growth conditions (PPFD 1000, 29 °C, 50 % relative humidity). The supply lines run from
the operating point to the ambient CO₂; the paper prints A = 21.1 (Ci = 25.9, gs = 370) and
A = 27.4 (Ci = 49.5, gs = 283) for plants grown at 35 Pa, A = 26.3 (Ci = 48.4, gs = 253) for
plants grown at 65 Pa.

In [ ]:
fig8 = {k: harley(Ci=np.linspace(5, 80, 376), **s) for k, s in {"35": SET35, "65": SET65}.items()}
OPERATING = {"35": [35.0, 65.0], "65": [65.0]}
points = {k: [operating_point(Ca, **(SET35 if k == "35" else SET65)) for Ca in cas] for k, cas in OPERATING.items()}
for k, pts in points.items():
    for Ca, pt in zip(OPERATING[k], pts):
        print(f"grown at {k} Pa, Ca {Ca:.0f}: A {pt['A']:.2f}, Ci {pt.name:.2f}, gs {pt['gs']:.0f}")
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
for ax, k, title in zip(axs, ["35", "65"], ["Cotton, grown at 35 Pa CO$_2$", "Grown at 65 Pa CO$_2$"]):
    ax.plot(fig8[k].index, fig8[k]["A"], color="black", linewidth=2)
    for Ca, pt in zip(OPERATING[k], points[k]):
        ax.plot([pt.name, Ca], [pt["A"], 0], color="black", linewidth=1)
        ax.plot(pt.name, pt["A"], "o", color="black")
    ax.set(title=title, xlabel="CO$_2$ Partial Pressure (Pa)", xlim=(0, 70 if k == "35" else 80), ylim=(0, 35))
axs[0].set_ylabel(r"Net Photosynthesis ($\mu$mol m$^{-2}$ s$^{-1}$)")
fig.tight_layout()
fig.savefig("figs/fig8.svg")
plt.show()